# 03- Implementación modelos de clasificación supervisada

**Proyecto Final Machine Learning**  
Adriana Lorenzo Mingorance

En este notebook vamos a construir distintos modelos de clasificación supervisada para predecir la variable objetivo: la situación final del estudiante (abandono, matriculado o graduado). 

Los modelos que vamos a implementar son:
1. Regresión logística
2. KNN
3. Random Forest 
4. Boosting


## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score

semilla = 42

## 1. Carga de datos

In [ ]:
X_train = pd.read_csv('../data/X_train_clf.csv')
X_test = pd.read_csv('../data/X_test_clf.csv')

X_train_sc = pd.read_csv('../data/X_train_clf_scaled.csv')
X_test_sc = pd.read_csv('../data/X_test_clf_scaled.csv')

y_train = pd.read_csv('../data/y_train_clf.csv')['objetivo']
y_test = pd.read_csv('../data/y_test_clf.csv')['objetivo'] #para convertirlo a una serie (vector 1D)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Clases: {y_train.value_counts().to_dict()}")


Train: (3539, 237) | Test: (885, 237)
Clases: {'graduado': 1767, 'abandono': 1137, 'matriculado': 635}


## 2. Análisis del desbalanceo de clases

In [3]:
porcentajes = y_train.value_counts(normalize=True)*100
print("Distribución en train: ")
for clase, p in porcentajes.items():
    print(f" {clase}: {p:.1f}%")

#ratio de desbalanceo
ratio = porcentajes.max() / porcentajes.min()
print(f"\nRatio max/min: {ratio:.1f}x")

if ratio > 1.5:
    print("Hay cierto desbalanceo; tendremos esto en cuenta en la evaluación")

Distribución en train: 
 graduado: 49.9%
 abandono: 32.1%
 matriculado: 17.9%

Ratio max/min: 2.8x
Hay cierto desbalanceo; tendremos esto en cuenta en la evaluación


## 3. Función auxiliar de evaluación de modelos

In [4]:
resultados = {} #para guardar métricas de todos los modelos

def evaluar_modelo(nombre, modelo, X_tr, X_test, y_tr, y_test):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_test)

    #usamos las metricas que hemos importdo de sklearn
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_pond = f1_score(y_test, y_pred, average='weighted')

    #hacemos validacion cruzada con k= 5 folds, sobre train
    cv_scores = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='f1_macro')

    #guardamos los resultados
    resultados[nombre] = {
        'accuracy': acc,
        'F1 macro': f1_macro,
        'F1 ponderado': f1_pond,
        'CV F1 macro (mean)': cv_scores.mean(),
        'CV F1 macro (std)': cv_scores.std()
    }

    #imprimimos todo:
    print("\nNombre: ")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 macro: {f1_macro:.4f}")
    print(f"F1 ponderado: {f1_pond:.4f}")
    print(f"CV F1 macro: {cv_scores.mean():.4f} +/- {cv_scores.mean():.4f}")
    print()
    print(classification_report(y_test, y_pred))

    return modelo, y_pred

## Modelo 1: Regresión logística

In [ ]:
reg_log = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=semilla)

#reg_log_fit, reg_log_pred =